In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import random_split

from torch_geometric.data import DataLoader, Data
from torch_geometric.datasets import MoleculeNet
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.nn import GCNConv, GINEConv 
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
import dgl
from dgl.nn import GINConv, SumPooling, GCN2Conv, GINEConv, GraphConv, GATConv
from dgllife.model import MPNNPredictor, GCN
from dgllife.utils import smiles_to_bigraph, CanonicalAtomFeaturizer, CanonicalBondFeaturizer
from dgl.dataloading import GraphDataLoader
from torch.utils.data import random_split

In [3]:
from torch_geometric.datasets import AQSOL
from torch_geometric.datasets import freebase
from torch_geometric.datasets import ZINC

In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem

In [5]:
df1 = pd.read_csv('./idotdo_final_871.csv', na_values=['na'])
df1 = df1.dropna(ignore_index=True)
df1

,CHEMBLID,smiles,ido_ic50,tdo_ic50
0,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,7.638272,7.221849
1,CHEMBL1209728,Cc1c(Br)oc2c1C(=O)C(=O)c1c-2ccc2c1CCCC2(C)C,5.000000,5.000000
2,CHEMBL1276265,O=C1c2ccc(Cl)cc2-n2c1nc1ccccc1c2=O,6.737549,3.649364
3,CHEMBL1346056,Oc1ccccc1-c1nc2c3ccccc3c3ccccc3c2[nH]1,4.928486,4.911864
4,CHEMBL139935,O=[N+]([O-])c1cc(F)c2cccnc2c1O,4.835350,4.519562
...,...,...,...,...
866,43a,CS(=O)(NC1=CC=C(C2=CC3=C(C=N2)C(O)=C(OC)C=C3)C...,5.906578,6.698970
867,43b,O=S(C(F)(F)F)(NC1=CC=C(C2=CC3=C(C=N2)C(O)=C(OC...,6.508638,7.096910
868,43c,O=S(C1CC1)(NC2=CC=C(C3=CC4=C(C=N3)C(O)=C(OC)C=...,5.342944,5.473661
869,43d,O=S(N1CCCCC1)(NC2=CC=C(C3=CC4=C(C=N3)C(O)=C(OC...,4.813326,5.098542


In [6]:
df3 = df1.loc[:, ['ido_ic50', 'tdo_ic50']]
df3

,ido_ic50,tdo_ic50
0,7.638272,7.221849
1,5.000000,5.000000
2,6.737549,3.649364
3,4.928486,4.911864
4,4.835350,4.519562
...,...,...
866,5.906578,6.698970
867,6.508638,7.096910
868,5.342944,5.473661
869,4.813326,5.098542


In [7]:
for i in range(760):
    if df3['ido_ic50'].values[i] >= 6.15:
        df3.loc[i, ['ido_ic50']] = 1
    else:
        df3.loc[i, ['ido_ic50']] = 0

In [8]:
for i in range(760):
    if df3['tdo_ic50'].values[i] >= 6.0:
        df3.loc[i, ['tdo_ic50']] = 1
    else:
        df3.loc[i, ['tdo_ic50']] = 0

In [9]:
df3

,ido_ic50,tdo_ic50
0,1.000000,1.000000
1,0.000000,0.000000
2,1.000000,0.000000
3,0.000000,0.000000
4,0.000000,0.000000
...,...,...
866,5.906578,6.698970
867,6.508638,7.096910
868,5.342944,5.473661
869,4.813326,5.098542


In [10]:
newcol = []
for i in range(len(df3)):
    if df3['ido_ic50'].values[i] == 1 and df3['tdo_ic50'].values[i] == 1:
        newcol.append('AA')
    elif df3['ido_ic50'].values[i] == 0 and df3['tdo_ic50'].values[i] == 0:
        newcol.append('II')
    elif df3['ido_ic50'].values[i] == 0 and df3['tdo_ic50'].values[i] == 1:
        newcol.append('IA')
    else:
        newcol.append('AI')

In [11]:
target = pd.DataFrame(data=newcol, columns=['ido_tdo'])
target

,ido_tdo
0,AA
1,II
2,AI
3,II
4,II
...,...
866,AI
867,AI
868,AI
869,AI


In [12]:
y = target.values.reshape(len(df3),)
y.shape

(871,)

In [13]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [14]:
y_label = encoder.fit_transform(y)

In [15]:
y_label

array([0, 3, 1, 3, 3, 3, 2, 3, 3, 3, 2, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1,
       3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 0, 2, 3, 3, 1, 2, 1, 2, 3, 3, 2, 3,
       3, 2, 2, 2, 2, 0, 2, 0, 2, 2, 2, 2, 2, 2, 2, 3, 1, 3, 3, 3, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 0, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1,
       3, 1, 1, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 2, 0, 1, 3, 0, 0,
       1, 3, 0, 2, 1, 3, 3, 1, 0, 1, 0, 0, 0, 0, 1, 3, 1, 0, 0, 0, 1, 0,
       3, 1, 0, 0, 1, 0, 2, 0, 0, 0, 1, 0, 0, 1, 1, 3, 0, 0, 3, 0, 0, 3,
       0, 1, 2, 1, 0, 0, 0, 0, 2, 2, 1, 0, 1, 0, 3, 0, 0, 0, 0, 0, 3, 0,
       0, 1, 1, 2, 0, 0, 0, 1, 0, 0, 3, 1, 1, 2, 0, 3, 3, 0, 0, 3, 0, 3,
       0, 3, 3, 3, 0, 0, 0, 1, 1, 1, 1, 0, 2, 0, 3, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 3, 0, 3, 3, 3, 2, 2, 0, 1, 0, 2, 3, 0, 0, 1, 0, 0, 0,
       3, 3, 0, 0, 3, 3, 1, 2, 3, 0, 3, 0, 2, 0, 1, 3, 1, 0, 0, 3, 3, 0,
       0, 2, 1, 0, 3, 0, 3, 3, 0, 3, 3, 1, 1, 3, 3,

In [16]:
from rdkit.Chem import SaltRemover as sr
remover = sr.SaltRemover()

In [17]:
mols = []
for i in range(len(df1)):
    try:
        mol_i = Chem.MolFromSmiles(df1['smiles'][i])
        if mol_i is None:
            print(f"[WARNING] Invalid SMILES at index {i}, skipping.")
            mols.append(None)  # Maintain indexing for multiprocessing
            continue
        mol_i = remover.StripMol(mol_i, dontRemoveEverything=True)
        mols.append(mol_i)
    except Exception as e:
        print(e)
len(mols)

871

In [ ]:
#DGL featurizer produces one-hot encode features. That is why feature dimension is higher than PyG features.

✅ Best solution: Use DGL featurizers inside PyG

In [19]:
#Step 1: Import DGL featurizers
from dgllife.utils import (
    CanonicalAtomFeaturizer,
    CanonicalBondFeaturizer
)
from rdkit import Chem
import torch
from torch_geometric.data import Data


In [20]:
#Prefer this over the manual atom and bond featurizer written inthe upper cells.
# Initialize once (do NOT recreate inside function)
atom_featurizer = CanonicalAtomFeaturizer()
bond_featurizer = CanonicalBondFeaturizer()


def mol_to_pyg_data_dgl_features(mol: Chem.Mol, y=None) -> Data:
    """
    Convert RDKit Mol into PyG Data object
    using DGL Canonical Atom & Bond featurizers.
    """

    # ===== Node features =====
    atom_feats_dict = atom_featurizer(mol)
    x = atom_feats_dict["h"].clone().detach().float()

    # ===== Bond features =====
    bond_feats_dict = bond_featurizer(mol)
    edge_feats = bond_feats_dict["e"].clone().detach().float()

    edge_index = []
    edge_attr = []

    # Iterate over bonds once
    for idx, bond in enumerate(mol.GetBonds()):
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        # Add both directions
        edge_index.append([i, j])
        edge_index.append([j, i])

        # Duplicate bond feature for both directions
        edge_attr.append(edge_feats[idx])
        edge_attr.append(edge_feats[idx])

    # Convert to tensors
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.stack(edge_attr, dim=0)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

    if y is not None:
        data.y = torch.tensor([y], dtype=torch.long)

    return data


In [ ]:
✅ Now PyG sees exactly the same 74 / 12 features as DGL

In [21]:
#Using DGL featurizer
def mol_to_molsData(mols):
    mols_list = []
    for mol in mols:
        data = mol_to_pyg_data_dgl_features(mol)
        mols_list.append(data)
    return mols_list

In [22]:
#Using DGL featurizer
mols_list = mol_to_molsData(mols)
for i, data in enumerate(mols_list):
    # Wrap each label as a tensor
    data.y = torch.tensor([y_label[i]], dtype=torch.long)

In [23]:
data = mol_to_pyg_data_dgl_features(mols[0])

print(data.x.shape)
print(data.edge_index.shape)
print(data.edge_attr.shape)


torch.Size([20, 74])
torch.Size([2, 46])
torch.Size([46, 12])


In [24]:
data.edge_index.shape[1] == data.edge_attr.shape[0]

True

In [25]:
len(mols_list)

871

In [26]:
#Now with considering each molecule as a graph, splitting the data becomes very simple
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

# Suppose your dataset is a list of Data objects, length 1126
dataset = mols_list  # already prepared

train_len = int(0.6 * len(dataset))
val_len   = int(0.2 * len(dataset))
test_len  = len(dataset) - train_len - val_len

generator1 = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_len, val_len, test_len], generator=generator1)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

In [27]:
len(train_dataset), len(val_dataset), len(test_dataset)

(522, 174, 175)

In [28]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d
from torch_geometric.nn import GINEConv, global_add_pool

class GINE(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_classes):
        super().__init__()
        torch.manual_seed(1234)

        def mlp(in_dim, out_dim):
            return Sequential(
                Linear(in_dim, out_dim),
                BatchNorm1d(out_dim),
                ReLU(),
                Linear(out_dim, out_dim)
            )

        # ----- GINE layers -----
        self.conv1 = GINEConv(
            mlp(num_node_features, 64),
            edge_dim=num_edge_features
        )

        self.conv2 = GINEConv(
            mlp(64, 64),
            edge_dim=num_edge_features
        )

        self.conv3 = GINEConv(
            mlp(64, 64),
            edge_dim=num_edge_features
        )

        # ----- Residual projections -----
        self.res1 = Linear(num_node_features, 64)
        self.res2 = Linear(64, 64)
        self.res3 = Linear(64, 64)

        # ----- Graph classifier -----
        self.classifier = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_attr, batch):
        # ===== Layer 1 =====
        h1 = self.conv1(x, edge_index, edge_attr)
        h1 = F.relu(h1 + self.res1(x))

        # ===== Layer 2 =====
        h2 = self.conv2(h1, edge_index, edge_attr)
        h2 = F.relu(h2 + self.res2(h1))

        # ===== Layer 3 =====
        h3 = self.conv3(h2, edge_index, edge_attr)
        h3 = F.relu(h3 + self.res3(h2))

        # ===== Graph-level pooling =====
        h_graph = global_add_pool(h3, batch)

        out = self.classifier(h_graph)
        return out


In [29]:
#In the above cell, mlp dimensions of 128 and 32 were tested and the reults were worse than 64.

In [30]:
num_node_features = dataset[0].x.shape[1]
num_edge_features = dataset[0].edge_attr.shape[1]

In [31]:
num_node_features, num_edge_features

(74, 12)

In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GINE(num_node_features=num_node_features,
             num_edge_features=num_edge_features,
             num_classes=4).to(device)

In [33]:
print(model)

GINE(
  (conv1): GINEConv(nn=Sequential(
    (0): Linear(in_features=74, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
  ))
  (conv2): GINEConv(nn=Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
  ))
  (conv3): GINEConv(nn=Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=64, bias=True)
  ))
  (res1): Linear(in_features=74, out_features=64, bias=True)
  (res2): Linear(in_features=64, out_features=64, bias=True)
  (res3): Linear(in_features=64, out_features=64, bias=True)
  (classifier): 

In [34]:
import torch
import torch.nn.functional as F
import os
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GINE(num_node_features=num_node_features,
             num_edge_features=num_edge_features,
             num_classes=4).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

# Scheduler: reduce LR by factor of 0.5 if val_acc doesn't improve for 5 epochs
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

# Early stopping parameters
patience = 10          # stop if no improvement for 10 epochs
#best_val_acc = 0.0
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_path = "best_gine_871.pth"

def train(loader):
    model.train()
    total_loss, correct = 0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        correct += (out.argmax(dim=1) == data.y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            loss = criterion(out, data.y)
            total_loss += loss.item() * data.num_graphs
            correct += (out.argmax(dim=1) == data.y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# Training loop with early stopping
num_epochs = 100
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train(train_loader)
    val_loss, val_acc = evaluate(val_loader)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f}")
    
    # Step scheduler based on validation accuracy
    scheduler.step(val_loss)
    
    # Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        # Save best model
        torch.save(model.state_dict(), best_model_path)
        print(f"--> Saved new best model at epoch {epoch} with Val Acc: {val_acc:.3f}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}. Best Val Acc: {val_acc:.3f}")
            break

/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 001 | Train Loss: 1.530 | Train Acc: 0.439 | Val Loss: 1.298 | Val Acc: 0.316
--> Saved new best model at epoch 1 with Val Acc: 0.316
Epoch 002 | Train Loss: 0.988 | Train Acc: 0.625 | Val Loss: 1.471 | Val Acc: 0.362
Epoch 003 | Train Loss: 0.818 | Train Acc: 0.680 | Val Loss: 1.285 | Val Acc: 0.540
--> Saved new best model at epoch 3 with Val Acc: 0.540
Epoch 004 | Train Loss: 0.760 | Train Acc: 0.701 | Val Loss: 1.006 | Val Acc: 0.638
--> Saved new best model at epoch 4 with Val Acc: 0.638
Epoch 005 | Train Loss: 0.672 | Train Acc: 0.734 | Val Loss: 1.519 | Val Acc: 0.529
Epoch 006 | Train Loss: 0.643 | Train Acc: 0.759 | Val Loss: 1.000 | Val Acc: 0.626
--> Saved new best model at epoch 6 with Val Acc: 0.626
Epoch 007 | Train Loss: 0.569 | Train Acc: 0.793 | Val Loss: 0.963 | Val Acc: 0.678
--> Saved new best model at epoch 7 with Val Acc: 0.678
Epoch 008 | Train Loss: 0.510 | Train Acc: 0.787 | Val Loss: 0.943 | Val Acc: 0.678
--> Saved new best model at epoch 8 with Val Acc

In [ ]:
# Load best model before testing
model.load_state_dict(torch.load(best_model_path, weights_only=None))

In [35]:
test_loss, test_acc = evaluate(test_loader)
print(f"Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.3f}")

Test Loss: 0.910 | Test Acc: 0.720


In [36]:
from sklearn.metrics import classification_report

In [85]:
#This is only for calculating classification report
def class_report(loader):
    model.eval()
    y_true, y_prediction =[], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            pred = out.argmax(dim=1)
            y_true.extend(data.y.cpu().numpy())
            y_prediction.extend(pred.cpu().numpy())
    return y_true, y_prediction

In [86]:
y_true, y_pred = class_report(test_loader)

In [87]:
np.unique(y_pred)

array([0, 1, 2, 3])

In [88]:
len(y_pred), len(y_true)

(175, 175)

In [89]:
# Print classification report
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.80      0.78        46
           1       0.85      0.74      0.79        53
           2       0.25      0.17      0.20        18
           3       0.69      0.81      0.75        58

    accuracy                           0.72       175
   macro avg       0.64      0.63      0.63       175
weighted avg       0.71      0.72      0.71       175



In [81]:
from sklearn.metrics import roc_curve, RocCurveDisplay, auc, roc_auc_score

In [92]:
def test_ROC(model, loader):
    model.eval()
    y_true, y_logits =[], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            y_true.extend(data.y.cpu().numpy())
            y_logits.extend(out.cpu().numpy())
    return y_true, y_logits

In [93]:
import torch.nn.functional as F

y_true, y_logits = test_ROC(model, test_loader)
y_logits = torch.tensor(np.array(y_logits))
proba = F.softmax(y_logits, dim=1)

In [95]:
roc_auc_score(y_true, proba, multi_class='ovr', average='macro')

0.8767434887690393

In [97]:
roc_auc_score(y_true, proba, multi_class='ovr', average=None)

array([0.94725312, 0.89065883, 0.79299363, 0.87606838])